## Sentiment Analysis

In this exercise we use the IMDb-dataset, which we will use to perform a sentiment analysis. The code below assumes that the data is placed in the same folder as this notebook. We see that the reviews are loaded as a pandas dataframe, and print the beginning of the first few reviews.

In [11]:
import numpy as np
import pandas as pd

reviews = pd.read_csv('reviews.txt', header=None)
labels = pd.read_csv('labels.txt', header=None)
Y = (labels=='positive').astype(np.int_)

print(type(reviews))
print(reviews.head())

<class 'pandas.core.frame.DataFrame'>
                                                   0
0  bromwell high is a cartoon comedy . it ran at ...
1  story of a man who has unnatural feelings for ...
2  homelessness  or houselessness as george carli...
3  airport    starts as a brand new luxury    pla...
4  brilliant over  acting by lesley ann warren . ...


In [18]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import classification_report, confusion_matrix

# tensorflow.keras is loaded lazily so the IDE can't find it.
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# fix that worked for me/not sure if its the same thing
# from keras.src import Sequential
# from keras.src.layers import Dense, Dropout

**(a)** Split the reviews and labels in test, train and validation sets. The train and validation sets will be used to train your model and tune hyperparameters, the test set will be saved for testing. Use the `CountVectorizer` from `sklearn.feature_extraction.text` to create a Bag-of-Words representation of the reviews. Only use the 10,000 most frequent words (use the `max_features`-parameter of `CountVectorizer`).

In [13]:
reviews_train, reviews_test, Y_train, Y_test = train_test_split(
    reviews[0],
    Y[0],
    test_size=0.2,
    random_state=42
)

reviews_train, reviews_val, Y_train, Y_val = train_test_split(
    reviews_train,
    Y_train,
    test_size=0.25,
    random_state=42
)

vectorizer = CountVectorizer(max_features=10000)

X_train = vectorizer.fit_transform(reviews_train)

X_val = vectorizer.transform(reviews_val)
X_test = vectorizer.transform(reviews_test)

**(b)** Explore the representation of the reviews. How is a single word represented? How about a whole review?

In [14]:
print("First 100 features:")
print(vectorizer.get_feature_names_out()[:100])

print("First Review:")

feature_names = vectorizer.get_feature_names_out()
first_review = reviews_train.iloc[0]
print(first_review)

print("Indexes:")
first_review_vector = X_train[0]

for idx in first_review_vector.indices:
    feature_name = feature_names[idx]
    count = first_review_vector[0, idx]
    print(f"The word '{feature_name}' appears {count} times in the first review.")

First 100 features:
['abandon' 'abandoned' 'abby' 'abc' 'abducted' 'abilities' 'ability'
 'able' 'aboard' 'abominable' 'abomination' 'abortion' 'abound' 'about'
 'above' 'abraham' 'abrupt' 'abruptly' 'absence' 'absent' 'absolute'
 'absolutely' 'absorbed' 'absorbing' 'abstract' 'absurd' 'absurdity' 'abu'
 'abundance' 'abuse' 'abused' 'abusive' 'abysmal' 'academic' 'academy'
 'accent' 'accents' 'accept' 'acceptable' 'acceptance' 'accepted'
 'accepting' 'accepts' 'access' 'accessible' 'accident' 'accidental'
 'accidentally' 'acclaim' 'acclaimed' 'accompanied' 'accompanying'
 'accomplish' 'accomplished' 'accomplishment' 'according' 'account'
 'accounts' 'accuracy' 'accurate' 'accurately' 'accusations' 'accused'
 'ace' 'achieve' 'achieved' 'achievement' 'achievements' 'achieves' 'acid'
 'acknowledge' 'acknowledged' 'acquire' 'acquired' 'across' 'act' 'acted'
 'acting' 'action' 'actions' 'active' 'activities' 'activity' 'actor'
 'actors' 'actress' 'actresses' 'acts' 'actual' 'actuality' 'act

**(c)** Train a neural network with a single hidden layer on the dataset, tuning the relevant hyperparameters to optimize accuracy. 

In [ ]:
X_train_dense = X_train.toarray()
X_val_dense = X_val.toarray()
X_test_dense = X_test.toarray()

input_dim = X_train_dense.shape[1]
model = Sequential() # 
model.add(Dense(128, input_dim=input_dim, activation='relu')) # hidden layer with 128 neurons
model.add(Dropout(0.5)) # regularization (some neurons are randomly dropped out of the network)
model.add(Dense(1, activation='sigmoid'))  # sigmoid for binary classification, softmax for multi-class

model.compile(loss='binary_crossentropy', 
              optimizer='adam',
              metrics=['accuracy'])

history = model.fit(X_train_dense, Y_train,
                    epochs=10,
                    verbose=True,
                    validation_data=(X_val_dense, Y_val),
                    batch_size=10)

loss, accuracy = model.evaluate(X_test_dense, Y_test, verbose=False)
print(f"Test Accuracy: {accuracy:.4f}")

/home/mark/.conda/envs/yoloai/lib/python3.12/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10


2025-04-21 13:30:03.905033: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 1200000000 exceeds 10% of free system memory.


1497/1500 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7917 - loss: 0.4542

2025-04-21 13:30:18.989657: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 400000000 exceeds 10% of free system memory.


1500/1500 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - accuracy: 0.7919 - loss: 0.4540 - val_accuracy: 0.8742 - val_loss: 0.3051
Epoch 2/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - accuracy: 0.9135 - loss: 0.2178 - val_accuracy: 0.8860 - val_loss: 0.2792
Epoch 3/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - accuracy: 0.9495 - loss: 0.1387 - val_accuracy: 0.8828 - val_loss: 0.2968
Epoch 4/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - accuracy: 0.9708 - loss: 0.0871 - val_accuracy: 0.8864 - val_loss: 0.3465
Epoch 5/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - accuracy: 0.9805 - loss: 0.0601 - val_accuracy: 0.8802 - val_loss: 0.3954
Epoch 6/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 19s 13ms/step - accuracy: 0.9832 - loss: 0.0526 - val_accuracy: 0.8794 - val_loss: 0.4101
Epoch 7/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 19s 13ms/step - accuracy: 0.9897 - loss: 0.0350 - val_accuracy: 0.8876 - val_loss: 0.4363
Epoch 8/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - accuracy: 0.9893 - loss: 0.03

2025-04-21 13:32:54.951218: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 400000000 exceeds 10% of free system memory.


Test Accuracy: 0.8790


**(d)** Test your sentiment-classifier on the test set.

In [16]:
loss, accuracy = model.evaluate(X_test_dense, Y_test, verbose=True)

print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")
predictions = model.predict(X_test_dense)

predicted_classes = (predictions > 0.5).astype(int)
print(classification_report(Y_test, predicted_classes, target_names=['Negative', 'Positive']))
confusion_matrix = confusion_matrix(Y_test, predicted_classes)
print(confusion_matrix)

157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8854 - loss: 0.6718
Test Loss: 0.6909
Test Accuracy: 0.8790
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
              precision    recall  f1-score   support

    Negative       0.89      0.86      0.88      2492
    Positive       0.87      0.90      0.88      2508

    accuracy                           0.88      5000
   macro avg       0.88      0.88      0.88      5000
weighted avg       0.88      0.88      0.88      5000

[[2143  349]
 [ 256 2252]]


**(e)** Use the classifier to classify a few sentences you write yourselves. 

In [ ]:
new_sentences = [
    "I did not like the movie, it was boring",
    "This movie was fantastic",
    "I really liked the movie",
    "A masterpiece of cinema",
    "The plot was predictable",
    "It was interesting movie, although very long",
    "Short movie with not an interesting outcome",
    "Did not like the movie",
    "I love my mom but hated the movie",
]

X_new = vectorizer.transform(new_sentences)
X_new_dense = X_new.toarray()
new_predictions = model.predict(X_new_dense)
new_predicted_classes = (new_predictions > 0.5).astype(int)

for i, sentence in enumerate(new_sentences):
    print(f"Sentence: '{sentence}'")
    print(f"Predicted Sentiment: {'Positive' if new_predicted_classes[i][0] == 1 else 'Negative'}\n")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
Sentence: 'I did not like the movie, it was boring'
Predicted Sentiment: Negative

Sentence: 'This movie was a fantastic'
Predicted Sentiment: Positive

Sentence: 'I really liked the movie'
Predicted Sentiment: Positive

Sentence: 'A masterpiece of cinema'
Predicted Sentiment: Positive

Sentence: 'The plot was predictable'
Predicted Sentiment: Negative

Sentence: 'It was interesting movie, although very long'
Predicted Sentiment: Negative

Sentence: 'Short movie with not an interesting outcome'
Predicted Sentiment: Negative

Sentence: 'Did not like the movie'
Predicted Sentiment: Negative

Sentence: 'I love my mom but hated the movie'
Predicted Sentiment: Negative

